In [0]:

import boto3
from botocore.config import Config

b2_access_key = "0030087346c1a0c0000000003"
b2_secret_key = "K003PBOFq+d3MQ8rvQXZ9SsW58Z+TDI"
endpoint = "s3.eu-central-003.backblazeb2.com"
bucket   = "rtv-lakehouse-raw"

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{endpoint}",
    aws_access_key_id=b2_access_key,
    aws_secret_access_key=b2_secret_key,
    config=Config(s3={"addressing_style": "path"}),
)

# list the files
response = s3.list_objects_v2(Bucket=bucket, Prefix="raw/")
for obj in response.get("Contents", []):
    print(obj["Key"])

In [0]:
# unzipping the datasets
import os
import zipfile
import io

local_dir = "/tmp/datasets"
os.makedirs(local_dir, exist_ok=True)

# Use the s3 client and bucket defined in the previous cell
response = s3.list_objects_v2(Bucket=bucket, Prefix="raw/")
for obj in response.get("Contents", []):
    key = obj["Key"]
    if key.endswith(".zip"):
        print(f"Downloading {key} ...")
        file_obj = s3.get_object(Bucket=bucket, Key=key)
        zip_bytes = file_obj["Body"].read()

        print(f"  Unzipping {key} ...")
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            zf.extractall(local_dir)
            for name in zf.namelist():
                print(f"    extracted: {name}")
    else:
        # download non-zip files as-is
        local_path = os.path.join(local_dir, os.path.basename(key))
        print(f"Downloading {key} -> {local_path}")
        s3.download_file(bucket, key, local_path)

print(f"\nAll files extracted to: {local_dir}")
for root, dirs, files in os.walk(local_dir):
    for f in files:
        print(os.path.join(root, f))


In [0]:
!ls

Processing and storing a bronze table for two UNPS csv: agsec1.csv and agsec2a.csv

In [0]:
import pandas as pd
from pyspark.sql import functions as F

pdf_agsec1 = pd.read_csv("/tmp/datasets/UGA_2019_UNPS_v03_M_CSV/Agric/agsec1.csv")
bronze_unps_df_1 = spark.createDataFrame(pdf_agsec1)
bronze_unps_df_1.show(3)

# adding metadata columns
bronze_unps_df_1 = bronze_unps_df_1\
                    .withColumn("_ingested_at", F.current_timestamp())\
                    .withColumn("_source_file", F.lit("UGA_2019_UNPS_v03_M_CSV/Agric/agsec1.csv"))\
                    .withColumn("_source_system", F.lit("UNPS"))
# creating the catalog and schema
spark.sql("CREATE CATALOG IF NOT EXISTS community_lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS community_lakehouse.bronze")
# saving the dataframe as a table
bronze_unps_df_1.write.mode("overwrite").saveAsTable("community_lakehouse.bronze.unps1")

In [0]:
import pandas as pd
from pyspark.sql import functions as F

pdf_agsec2 = pd.read_csv("/tmp/datasets/UGA_2019_UNPS_v03_M_CSV/Agric/agsec2a.csv")
bronze_unps_df_2 = spark.createDataFrame(pdf_agsec2)
bronze_unps_df_2.show(3)

# adding metadata columns
bronze_unps_df_2 = bronze_unps_df_2\
                    .withColumn("_ingested_at", F.current_timestamp())\
                    .withColumn("_source_file", F.lit("UGA_2019_UNPS_v03_M_CSV/Agric/agsec2a.csv"))\
                    .withColumn("_source_system", F.lit("UNPS"))

# saving the dataframe as a table
bronze_unps_df_2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("community_lakehouse.bronze.unps2")

In [0]:
%%sql
SHOW TABLES IN community_lakehouse.bronze;

Processing the FAOSTAT dataset into a bronze table

In [0]:
pdf_faostat = pd.read_csv("/tmp/datasets/Production_Crops_Livestock_E_Africa.csv")
# cleaning the column names
pdf_faostat.columns = pdf_faostat.columns.str.replace(r"[ ,;{}()\n\t=]", "_", regex=True)
bronze_faostat_df = spark.createDataFrame(pdf_faostat)

# adding metadata columns
bronze_faostat_df = bronze_faostat_df\
                    .withColumn("_ingested_at", F.current_timestamp())\
                    .withColumn("_source_file", F.lit("Production_Crops_Livestock_E_Africa.csv"))\
                    .withColumn("_source_system", F.lit("FAOSTAT"))\
                    .filter("Area = 'Uganda'")
bronze_faostat_df.show(5)
# saving the dataframe as a table
bronze_faostat_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("community_lakehouse.bronze.faostat")

In [0]:
%sql
DESCRIBE HISTORY community_lakehouse.bronze.faostat

Processing CHIRPS NetCDF data from .nc 

In [0]:
%pip install xarray netcdf4

In [0]:
import xarray as xr
import pandas as pd
from pyspark.sql import functions as F

filepath = "/tmp/datasets/6d6bc493-0a8f-49a9-a25a-5c3b0664bd0e.nc"

# Open the NetCDF file with xarray
ds = xr.open_dataset(filepath)
print(ds)

# Convert to a flat DataFrame
df = ds.to_dataframe().reset_index()
print("\nDataFrame shape:", df.shape)
print(df.head())

# Create a Spark DataFrame from the pandas DataFrame
bronze_chirps_df = spark.createDataFrame(df)
bronze_chirps_df.show(5)
print(bronze_chirps_df.printSchema())

# Add metadata columns
bronze_chirps_df = bronze_chirps_df\
    .withColumn("_ingested_at", F.current_timestamp())\
    .withColumn("_source_file", F.lit("6d6bc493-0a8f-49a9-a25a-5c3b0664bd0e.nc"))\
    .withColumn("_source_system", F.lit("CHIRPS"))

# Save as a bronze table
bronze_chirps_df.write.mode("overwrite").option("overwriteSchema", "true")\
    .saveAsTable("community_lakehouse.bronze.chirps")


Confirming all the delta tables were added

In [0]:
%sql
SHOW TABLES IN community_lakehouse.bronze;